In [ ]:
import pandas as pd  
import numpy as np  
import matplotlib.pyplot as plt  
import seaborn as sns
from sklearn.cluster import KMeans  
from sklearn.preprocessing import StandardScaler  

# load and explore data

In [ ]:
df = pd.read_csv('iris.csv')

In [ ]:
df.info()

In [ ]:
df.head()

In [ ]:
# divide the dataset into three dataframes based on the species
setosa_df = df.query('species == "Iris-setosa"')
versicolor_df = df.query('species == "Iris-versicolor"')
virginica_df = df.query('species == "Iris-virginica"')

fig, axes = plt.subplots(nrows=1, ncols=2)

# Plot a scatter chart using x='sepal_length', y='sepal_width', and separate colors for each of the three dataframes
ax = setosa_df.plot.scatter(x='sepal_length', y='sepal_width', label='setosa', ax=axes[0])
ax = versicolor_df.plot.scatter(x='sepal_length', y='sepal_width', label='versicolor', color='green', ax=ax)
ax = virginica_df.plot.scatter(x='sepal_length', y='sepal_width', label='virginica', color='red', ax=ax)

# Plot a scatter chart using x='petal_length', y='petal_width', and separate colors for each of the three dataframes
ax = setosa_df.plot.scatter(x='petal_length', y='petal_width', label='setosa', ax=axes[1])
ax = versicolor_df.plot.scatter(x='petal_length', y='petal_width', label='versicolor', color='green', ax=ax)
ax = virginica_df.plot.scatter(x='petal_length', y='petal_width', label='virginica', color='red', ax=ax)

plt.show()

# prepare data

In [ ]:
#features = df[["sepal_length", "sepal_width"]].copy()
#features = df[["petal_length", "petal_width"]].copy()
features = df[["sepal_length", "sepal_width", "petal_length", "petal_width"]].copy()
features.head()

features_scaled_iris = StandardScaler().fit_transform(features)
features_scaled_iris

# k-means

In [ ]:
num_clusters =3
#kmeans_model = KMeans(n_clusters=3, random_state=0, n_init="auto")
kmeans_model = KMeans(n_clusters=3, random_state=0, init="k-means++")
cluster_labels = kmeans_model.fit_predict(features_scaled_iris)

In [ ]:
# Visualising the clusters  
for cluster in range(num_clusters):
    mask = cluster_labels == cluster
    plt.scatter(features_scaled_iris[mask, 0],features_scaled_iris[mask, 1],s=100,label=f"Cluster {cluster}")
plt.scatter(kmeans_model.cluster_centers_[:, 0], kmeans_model.cluster_centers_[:,1], s = 100, c = 'yellow', label = 'Centroids')
plt.legend()

# find the optimal number of cluster using the Elbow method

In [ ]:
distortions = []  
K = range(1,8)  
for k in K:  
 kmeans_model = KMeans(n_clusters=k)  
 kmeans_model.fit(features_scaled_iris)  
 distortions.append(kmeans_model.inertia_)# Plotting the distortions  
plt.figure(figsize=(16,8))  
plt.plot(K, distortions, 'bx-')  
plt.xlabel('k')  
plt.ylabel('Distortion')  
plt.title('The Elbow Method showing the optimal clusters')  
plt.show()

# DBSCAN

In [ ]:
from sklearn.datasets import make_moons, make_blobs
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors

In [ ]:
features, _ = make_moons(n_samples=200, noise=0.05, random_state=42)
#features_scaled, _  = make_blobs(n_samples=300, centers=3, n_features=2, cluster_std=1.0, random_state=42)

# Visualize the dataset
plt.figure(figsize=(10, 6))
plt.scatter(features[:, 0], features[:, 1])
plt.title('Moon-shaped Dataset')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.show()

In [ ]:
num_clusters =3
#kmeans_model = KMeans(n_clusters=3, random_state=0, n_init="auto")
kmeans_model = KMeans(n_clusters=3, random_state=0, init="k-means++")
cluster_labels = kmeans_model.fit_predict(features)

# Visualising the clusters  
for cluster in range(num_clusters):
    mask = cluster_labels == cluster
    plt.scatter(features[mask, 0],features[mask, 1],s=100,label=f"Cluster {cluster}")
plt.scatter(kmeans_model.cluster_centers_[:, 0], kmeans_model.cluster_centers_[:,1], s = 100, c = 'yellow', label = 'Centroids')
plt.legend()

In [ ]:
# plot k-distance graph

k=5

neigh = NearestNeighbors(n_neighbors=k)
neigh.fit(features)
distances, _ = neigh.kneighbors(features)
distances = np.sort(distances[:, k-1])
plt.figure(figsize=(10, 6))
plt.plot(distances)
plt.xlabel('Points')
plt.ylabel(f'{k}-th nearest neighbor distance')
plt.title('K-distance Graph')
plt.show()


# DBSCAN clustering

In [ ]:
epsilon = 0.15  # Chosen based on k-distance graph
min_samples = 5  # 2 * num_features (2D data)

dbscan = DBSCAN(eps=epsilon, min_samples=min_samples)
clusters = dbscan.fit_predict(features)

In [ ]:
# Visualize the results
plt.figure(figsize=(10, 6))
scatter = plt.scatter(features[:, 0], features[:, 1], c=clusters, cmap='viridis')
plt.colorbar(scatter)
plt.title('DBSCAN Clustering Results')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.show()

# Hierarchal Clustering 

In [ ]:
import scipy.cluster.hierarchy as sch  
from sklearn.cluster import AgglomerativeClustering

In [ ]:
plt.figure(figsize=(8, 5))  
dendrogram = sch.dendrogram(sch.linkage(features_scaled_iris, method='ward'), no_labels=True)
plt.title('Dendrogram')  
plt.xlabel('Features')  
plt.ylabel('Euclidean Distance')  
plt.show()

In [ ]:
num_clusters =3
model = AgglomerativeClustering(n_clusters=num_clusters, linkage="ward")
cluster_labels = model.fit_predict(features_scaled_iris)

for cluster in range(num_clusters):
    mask = cluster_labels == cluster
    plt.scatter(features_scaled_iris[mask, 2],features_scaled_iris[mask, 3],s=100,label=f"Cluster {cluster}")
plt.legend()
